# AF2·08 — Recycling & Confidence

**Mechanism of the day:** two finishing pieces that turn a folder into AlphaFold.
**Recycling** — run the whole network several times, feeding each prediction back in as
extra input, so it can refine its own answer. And the **confidence heads** — pLDDT and
PAE — which let the model tell you *which parts of its prediction to trust*, arguably the
single most useful thing AlphaFold outputs.

Recycling is disarmingly simple: after a full forward pass, take the predicted structure,
summarise it (as a distogram of the predicted `Cα`s), and add it to the inputs of another
pass. Three or four rounds and the network has effectively iterated on its own work — a
cheap way to add depth without adding parameters.

The confidence heads answer a different question. AlphaFold does not just predict a
structure; it predicts **how wrong each part of that structure probably is**:

- **pLDDT** — a per-residue confidence. High means "this residue is placed accurately";
  low flags the disordered loops and uncertain regions. It is trained to predict the
  model's own local accuracy.
- **PAE** (predicted aligned error) — a per-*pair* confidence: "if I align on residue
  `i`, how far off is residue `j`?" This is what tells you whether two domains are
  confidently positioned *relative to each other*, even when each is individually fine.

You will build the recycling loop and both confidence targets, train the heads alongside
the folder, and verify the payoff that makes AlphaFold trustworthy in practice: **its
confidence actually predicts its error** — the residues it calls high-confidence really
are the accurate ones.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions at
the bottom. Trains in ~40s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK, RED = '#2a78d6', '#008300', '#52514e', '#e34948'
L = 16; c = 48; cz = 24; NITER = 6

# ---- toy proteins + structure-module machinery (all from rungs 05-07, given) ----
def make_structure():
    pos = [np.zeros(3)]; d = rng.normal(0, 1, 3); d /= np.linalg.norm(d)
    for _ in range(L - 1):
        d = d + rng.normal(0, 0.5, 3); d /= np.linalg.norm(d); pos.append(pos[-1] + d - 0.03 * pos[-1])
    X = np.array(pos); X -= X.mean(0); return X.astype(np.float32)
DIST_BINS = torch.linspace(1.5, 9.0, 15)
def disto_np(X):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1)); return np.digitize(D, DIST_BINS.numpy())
def disto_from_coords(t):
    return torch.bucketize(torch.cdist(t, t), DIST_BINS).clamp(max=15)
Xtr = torch.tensor(np.array([make_structure() for _ in range(200)]))
Dtr = torch.tensor(np.array([disto_np(x.numpy()) for x in Xtr]))
Xte = torch.tensor(np.array([make_structure() for _ in range(40)]))
Dte = torch.tensor(np.array([disto_np(x.numpy()) for x in Xte]))

def hat(w):
    O = torch.zeros(*w.shape[:-1], 3, 3)
    O[...,0,1]=-w[...,2];O[...,0,2]=w[...,1];O[...,1,0]=w[...,2];O[...,1,2]=-w[...,0];O[...,2,0]=-w[...,1];O[...,2,1]=w[...,0]
    return O
def so3_exp(w):
    th = w.norm(dim=-1, keepdim=True).clamp_min(1e-8); K = hat(w / th); th = th[..., None]
    return torch.eye(3).expand_as(K) + torch.sin(th)*K + (1-torch.cos(th))*(K@K)
def true_frames(X):
    R = torch.zeros(L, 3, 3)
    for i in range(L):
        a = X[min(i+1,L-1)]-X[i]; b = X[max(i-1,0)]-X[i]
        e1=a/(a.norm()+1e-8); e2=b-(e1@b)*e1; e2=e2/(e2.norm()+1e-8)
        R[i]=torch.stack([e1,e2,torch.cross(e1,e2,dim=-1)],-1)
    return R
Rtr=[true_frames(Xtr[i]) for i in range(len(Xtr))]; Rte=[true_frames(Xte[i]) for i in range(len(Xte))]
def fape(tp,Rp,tt,Rt,clamp=10.):
    lp=torch.einsum('lji,lkj->lki',Rp,tp[None]-tp[:,None]); lt=torch.einsum('lji,lkj->lki',Rt,tt[None]-tt[:,None])
    return (lp-lt).norm(dim=-1).clamp(max=clamp).mean()

h,dd,Np,Npv=4,8,4,6
class IPA(nn.Module):
    def __init__(s):
        super().__init__(); s.qs=nn.Linear(c,h*dd);s.ks=nn.Linear(c,h*dd);s.vs=nn.Linear(c,h*dd)
        s.qp=nn.Linear(c,h*Np*3);s.kp=nn.Linear(c,h*Np*3);s.vp=nn.Linear(c,h*Npv*3);s.bz=nn.Linear(cz,h);s.gamma=nn.Parameter(torch.zeros(h));s.out=nn.Linear(h*dd+h*cz+h*Npv*3+h*Npv,c)
    def forward(s,x,z,R,t):
        qs=s.qs(x).view(L,h,dd);ks=s.ks(x).view(L,h,dd);vs=s.vs(x).view(L,h,dd)
        g=lambda lp:torch.einsum('lij,lhpj->lhpi',R,lp)+t[:,None,None,:]
        Qg=g(s.qp(x).view(L,h,Np,3));Kg=g(s.kp(x).view(L,h,Np,3));Vg=g(s.vp(x).view(L,h,Npv,3))
        scal=torch.einsum('ihd,jhd->ijh',qs,ks)/math.sqrt(dd);d2=((Qg[:,None]-Kg[None])**2).sum(-1).sum(-1)
        a=F.softmax(scal+s.bz(z)-0.5*F.softplus(s.gamma)[None,None]*d2,1)
        o_s=torch.einsum('ijh,jhd->ihd',a,vs).reshape(L,h*dd);o_z=torch.einsum('ijh,ijz->ihz',a,z).reshape(L,h*cz)
        og=torch.einsum('ijh,jhpx->ihpx',a,Vg);ol=torch.einsum('lji,lhpj->lhpi',R,og-t[:,None,None,:])
        return s.out(torch.cat([o_s,o_z,ol.reshape(L,h*Npv*3),ol.norm(dim=-1).reshape(L,h*Npv)],-1))
print('structure-module machinery loaded (rungs 05-07).')

## Part 1 — recycling

After a full pass produces a structure, recycling feeds a summary of it back into the
next pass. The summary AlphaFold uses is the **distogram of the predicted `Cα`
positions** — a compact, pose-invariant description of "the structure I just built".
It is added (as an extra embedding) to the input, and the network runs again.

Crucially, the recycled structure is **detached** from the gradient: each pass is trained
to improve on the *fixed* previous answer, which keeps training stable and memory low
(you do not backpropagate through all the passes at once).

### Rep 1 — `recycle_distogram(t_pred)`
Given predicted `Cα` coordinates `t_pred` `[L,3]`, return the `[L,L]` distogram bin
indices of the predicted structure, using `torch.cdist` and `torch.bucketize` against
`DIST_BINS` (clamp the max bin to 15). Detach it — it is a fixed input to the next pass.

In [ ]:
def recycle_distogram(t_pred):
    '''Distogram-bin summary [L,L] of a predicted structure, detached for recycling.'''
    # YOUR CODE HERE
    # hint: torch.bucketize(torch.cdist(t_pred, t_pred), DIST_BINS).clamp(max=15).detach()
    raise NotImplementedError

# --- checkpoint ---
rc = recycle_distogram(Xte[0])
assert rc.shape == (L, L) and rc.dtype == torch.long
assert (torch.diag(rc) == 0).all(), 'a residue is at distance 0 from itself (bin 0)'
assert not rc.requires_grad, 'the recycled feature must be detached'
# it matches the precomputed distogram of that structure
assert (rc == disto_from_coords(Xte[0])).all()
print('recycle summary ok — the predicted structure, encoded to feed back in ✓')

In [ ]:
class AlphaFoldLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.zin = nn.Embedding(16, cz); self.zrec = nn.Embedding(17, cz)   # 17th = "no recycle yet"
        self.sinit = nn.Parameter(torch.zeros(c)); self.pos = nn.Embedding(L, c)
        self.ipa = IPA(); self.ln = nn.LayerNorm(c); self.head = nn.Linear(c, 6)
        self.plddt = nn.Linear(c, 1)          # per-residue confidence head
        self.pae = nn.Linear(cz, 1)           # per-pair confidence head
    def one_pass(self, dbins, rec_bins):
        z = self.zin(dbins) + self.zrec(rec_bins)
        x = self.sinit[None].expand(L, c) + self.pos(torch.arange(L))
        R = torch.eye(3).expand(L, 3, 3).contiguous(); t = torch.zeros(L, 3); traj = []
        for _ in range(NITER):
            x = self.ln(x + self.ipa(x, z, R, t))
            u = self.head(x); R = R @ so3_exp(u[:, :3] * 0.3); t = t + torch.einsum('lij,lj->li', R, u[:, 3:])
            traj.append((R, t))
        return R, t, traj, x, z
    def forward(self, dbins, n_recycle=3):
        rec = torch.full((L, L), 16, dtype=torch.long)     # nothing recycled yet
        outs = []
        for _ in range(n_recycle):
            R, t, traj, x, z = self.one_pass(dbins, rec)
            outs.append((R, t, traj, x, z))
            rec = recycle_distogram(t)                     # feed this structure into the next pass
        return outs
print('AlphaFold-lite: structure module + recycling + two confidence heads.')

## Part 2 — the confidence targets

The confidence heads are trained to predict the model's *own* error. So we need to define
that error — and it must be a **local, pose-invariant** quantity, computed just like FAPE.

- **pLDDT target (per residue).** For residue `i`, average over all `j` the distance
  between where `j` lands (in `i`'s frame) in the prediction versus the truth. Small =
  residue `i` sees the rest of the structure correctly = high confidence.
- **PAE target (per pair).** For the ordered pair `(i, j)`, the distance between predicted
  and true position of `j` **in residue `i`'s frame**. This is directional: `PAE[i,j]`
  asks "aligned on `i`, how wrong is `j`?".

(Real AlphaFold predicts these as binned distributions and reports pLDDT on a 0–100 scale
where higher is better; we regress the error directly and keep "higher error = lower
confidence" explicit, which is the same information.)

### Rep 2 — `pae_error(t_pred, R_pred, t_true, R_true)`
Return the `[L, L]` matrix of per-pair aligned errors: for each `(i, j)`, the distance
between the predicted and true position of `Cα_j` expressed in frame `i`. (This is the
PAE target; the pLDDT target is its row-mean.)

In [ ]:
def pae_error(t_pred, R_pred, t_true, R_true):
    '''Per-pair aligned error [L,L]: distance between pred and true position of j in frame i.'''
    # YOUR CODE HERE
    # hint: loc_p = einsum('lji,lkj->lki', R_pred, t_pred[None]-t_pred[:,None])
    #       loc_t = einsum('lji,lkj->lki', R_true, t_true[None]-t_true[:,None])
    #       return (loc_p - loc_t).norm(dim=-1)
    raise NotImplementedError

# --- checkpoint ---
e_self = pae_error(Xte[0], Rte[0], Xte[0], Rte[0])
assert e_self.shape == (L, L) and e_self.max() < 1e-4, 'zero aligned error against itself'
Xw = torch.tensor(make_structure())
e_wrong = pae_error(Xw, true_frames(Xw), Xte[0], Rte[0])
assert e_wrong.mean() > 0.5, 'a wrong structure has large aligned error'
# the pLDDT target is the per-residue mean of the PAE
plddt_target = e_wrong.mean(1)
assert plddt_target.shape == (L,)
print('confidence targets ok — PAE is per-pair, pLDDT is its per-residue average ✓')

In [ ]:
net = AlphaFoldLite(); opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
t0 = time.time(); hist = []
for step in range(1200):
    i = int(rng.integers(len(Xtr))); outs = net(Dtr[i], n_recycle=3)
    loss = sum(sum(fape(tt, RR, Xtr[i], Rtr[i]) for RR, tt in traj) / len(traj)
               for (RR_, tt_, traj, x, z) in outs)
    # train the confidence heads on the final pass against the realized error (detached)
    R, t, traj, x, z = outs[-1]
    with torch.no_grad():
        pae_t = pae_error(t, R, Xtr[i], Rtr[i]); plddt_t = pae_t.mean(1)
    loss = loss + F.smooth_l1_loss(net.plddt(x).squeeze(-1), plddt_t) \
                + F.smooth_l1_loss(net.pae(z).squeeze(-1), pae_t)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: hist.append((step, loss.item()))
print('trained in %.0fs' % (time.time() - t0))

## Part 3 — does recycling help, and does confidence predict error?

Two measurements. First, run the trained model with 1, 2, 3 recycles and check FAPE
improves. (In this toy the gain is small — the input distogram already nearly determines
the fold — but it is real and monotonic. Recycling's big wins come at scale, where each
pass re-reads a genuinely improved representation.) Second, the important one: do the
confidence heads predict the truth?

### Rep 3 — `pearson(a, b)`
Plain Pearson correlation between two 1-D tensors. We will use it to ask whether predicted
confidence tracks realized error.

In [ ]:
def pearson(a, b):
    '''Pearson correlation coefficient between two 1-D tensors.'''
    # YOUR CODE HERE
    # hint: center both, then (a*b).sum() / (a.norm() * b.norm())
    raise NotImplementedError

# --- checkpoint ---
x_ = torch.randn(100); assert abs(pearson(x_, x_).item() - 1.0) < 1e-5 and abs(pearson(x_, -x_).item() + 1.0) < 1e-5
net.eval()
with torch.no_grad():
    for nr in (1, 2, 3):
        fs = [fape(net(Dte[i], n_recycle=nr)[-1][1], net(Dte[i], n_recycle=nr)[-1][0], Xte[i], Rte[i]).item()
              for i in range(len(Xte))]
        print('recycles = %d :  held-out FAPE %.3f' % (nr, np.mean(fs)))
    pl_p, pl_t, pae_p, pae_t = [], [], [], []
    for i in range(len(Xte)):
        R, t, traj, x, z = net(Dte[i], n_recycle=3)[-1]
        et = pae_error(t, R, Xte[i], Rte[i])
        pl_p.append(net.plddt(x).squeeze(-1)); pl_t.append(et.mean(1))
        pae_p.append(net.pae(z).squeeze(-1).flatten()); pae_t.append(et.flatten())
    pl_p, pl_t = torch.cat(pl_p), torch.cat(pl_t); pae_p, pae_t = torch.cat(pae_p), torch.cat(pae_t)
rho_plddt = pearson(pl_p, pl_t).item(); rho_pae = pearson(pae_p, pae_t).item()
assert rho_plddt > 0.3 and rho_pae > 0.3, 'confidence heads must correlate with real error'
print('\npLDDT head vs true per-residue error: Pearson %.2f' % rho_plddt)
print('PAE   head vs true per-pair error:    Pearson %.2f' % rho_pae)
print('The model predicts where it is wrong — that is what makes AF2 usable. ✓')

In [ ]:
# The calibration picture: sort residues by predicted confidence, show true error falls.
order = torch.argsort(pl_p)                        # low predicted error (= high confidence) first
q = len(order) // 4
groups = ['most\nconfident', '2nd', '3rd', 'least\nconfident']
bucket_err = [pl_t[order[k*q:(k+1)*q]].mean().item() for k in range(4)]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].scatter(pl_p.numpy(), pl_t.numpy(), s=8, alpha=.4, color=BLUE)
axes[0].set_xlabel('predicted error (pLDDT head)'); axes[0].set_ylabel('true per-residue error')
axes[0].set_title('pLDDT predicts accuracy (rho=%.2f)' % rho_plddt, fontsize=10); axes[0].grid(alpha=.15)
axes[1].bar(groups, bucket_err, color=[GREEN, '#7bb37b', '#d9a441', RED])
axes[1].set_ylabel('actual mean error'); axes[1].set_title('confident residues really are accurate', fontsize=10); axes[1].grid(alpha=.15, axis='y')
i0 = 0
with torch.no_grad():
    R, t, traj, x, z = net(Dte[i0], n_recycle=3)[-1]
    im = axes[2].imshow(net.pae(z).squeeze(-1).numpy(), cmap='viridis_r')
axes[2].set_title('predicted PAE[i,j]', fontsize=10); axes[2].set_xlabel('residue j'); axes[2].set_ylabel('residue i (aligned on)')
fig.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.show()

## Reflection — what just transferred

- **Recycling is iteration for free:** summarise the prediction (as a distogram of the
  predicted `Cα`s), add it back to the inputs, run again — detached, so training stays
  cheap and stable. Modest here, decisive at scale.
- **AlphaFold predicts its own error.** The confidence heads are trained against a
  local, pose-invariant error computed exactly like FAPE — pLDDT per residue, PAE per
  ordered pair.
- **pLDDT** flags which residues are placed well; **PAE** flags which *pairs/domains* are
  positioned well relative to each other (the directional, aligned-error view). Together
  they are why you can look at an AlphaFold model and know which parts to believe.
- You verified the thing that matters: the confidence **correlates with the truth**, and
  the residues the model calls confident really are the accurate ones. A predictor that
  knows when it is wrong is worth far more than one that is silently overconfident.

**Next rung:** `AF2·09 — End-to-end toy AlphaFold`. We connect Track A (the Evoformer
trunk, rungs 01–04) to Track B (the structure module, rungs 05–08) into a single model
that goes **MSA → structure**, trained end-to-end, and fold a toy protein from its
sequence alignment alone.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def recycle_distogram(t_pred):
    return torch.bucketize(torch.cdist(t_pred, t_pred), DIST_BINS).clamp(max=15).detach()

def pae_error(t_pred, R_pred, t_true, R_true):
    loc_p = torch.einsum('lji,lkj->lki', R_pred, t_pred[None] - t_pred[:, None])
    loc_t = torch.einsum('lji,lkj->lki', R_true, t_true[None] - t_true[:, None])
    return (loc_p - loc_t).norm(dim=-1)

def pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    return (a * b).sum() / (a.norm() * b.norm() + 1e-9)

print('reference solutions loaded — re-run the checkpoint cells above')